# Pipeline Machine Learning — Prediksi Produksi Padi Sumatera

**Dokumentasi reproducible pipeline ML berbasis Random Forest Regressor**

Notebook ini mendokumentasikan keseluruhan alur pemodelan *machine learning* yang digunakan dalam aplikasi Web GIS *Prediksi Produksi Padi Regional Sumatera Berdasarkan Faktor Iklim*, mulai dari pemuatan data, eksplorasi, pra-pemrosesan, pelatihan model, evaluasi, hingga contoh proyeksi skenario iklim.

Logika di notebook ini mencerminkan modul `data_pipeline.py` dan `ml_engine.py` pada aplikasi, namun disajikan secara linear dan eksploratif agar mudah ditelusuri dan direproduksi.

---
**Penulis:** Surya Andika — Program Studi Informatika, Universitas Andalas  
**Algoritma:** Random Forest Regressor (`scikit-learn`)  
**Target:** `Produksi` (ton)  
**Fitur:** `Luas Panen`, `Curah hujan`, `Kelembapan`, `Suhu rata-rata`, `Provinsi`


## 1. Import Library

Mengimpor seluruh pustaka yang dibutuhkan untuk analisis data, pemodelan, dan visualisasi.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Library berhasil diimpor.")

## 2. Pemuatan Data

Memuat dataset historis produksi padi Sumatera (1993–2020). Dataset memuat data tahunan untuk delapan provinsi di Sumatera.


In [ ]:
CSV_PATH = "../Data_Tanaman_Padi_Sumatera_version_1.csv"

df = pd.read_csv(CSV_PATH)
print(f"Dimensi dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
df.head()

### 2.1 Informasi Struktur Data

In [ ]:
df.info()

In [ ]:
# Daftar provinsi yang tersedia
print("Provinsi dalam dataset:")
for p in sorted(df["Provinsi"].unique()):
    print(" -", p)

## 3. Eksplorasi Data (EDA)

Memahami karakteristik statistik dan pola data sebelum pemodelan.


In [ ]:
# Statistik deskriptif fitur numerik
fitur_numerik = ["Produksi", "Luas Panen", "Curah hujan", "Kelembapan", "Suhu rata-rata"]
df[fitur_numerik].describe()

### 3.1 Pengecekan Nilai Hilang

In [ ]:
missing = df.isnull().sum()
print("Jumlah nilai hilang per kolom:")
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada nilai hilang.")

### 3.2 Tren Produksi Padi per Provinsi (1993–2020)

In [ ]:
plt.figure(figsize=(11, 6))
for prov in sorted(df["Provinsi"].unique()):
    sub = df[df["Provinsi"] == prov].sort_values("Tahun")
    plt.plot(sub["Tahun"], sub["Produksi"], marker="o", markersize=3, label=prov)

plt.title("Tren Produksi Padi 8 Provinsi Sumatera (1993-2020)", fontsize=13)
plt.xlabel("Tahun")
plt.ylabel("Produksi (ton)")
plt.legend(fontsize=8, ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Matriks Korelasi Antar Variabel

Melihat hubungan linear awal antara faktor iklim, luas panen, dan produksi.


In [ ]:
corr = df[fitur_numerik].corr()

plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap="YlGn", vmin=-1, vmax=1)
plt.colorbar(label="Koefisien Korelasi")
plt.xticks(range(len(fitur_numerik)), fitur_numerik, rotation=45, ha="right")
plt.yticks(range(len(fitur_numerik)), fitur_numerik)
for i in range(len(fitur_numerik)):
    for j in range(len(fitur_numerik)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.title("Matriks Korelasi", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Pra-Pemrosesan Data

Tahapan pra-pemrosesan mengikuti modul `data_pipeline.py`:

1. **Penanganan outlier** menggunakan metode Interquartile Range (IQR) dengan ambang 3×IQR (di-*cap*, bukan dibuang, agar tidak kehilangan observasi).
2. **Label Encoding** pada variabel kategorikal `Provinsi`.
3. **Normalisasi** fitur prediktor dan target menggunakan `MinMaxScaler`.


### 4.1 Penanganan Outlier (IQR Capping)

In [ ]:
df_clean = df.copy()

def cap_outlier_iqr(series, k=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    return series.clip(lower, upper)

for col in ["Curah hujan", "Kelembapan", "Suhu rata-rata"]:
    df_clean[col] = cap_outlier_iqr(df_clean[col])

print("Outlier capping selesai untuk fitur iklim (Curah hujan, Kelembapan, Suhu rata-rata).")

### 4.2 Label Encoding Provinsi

In [ ]:
le = LabelEncoder()
df_clean["Provinsi_enc"] = le.fit_transform(df_clean["Provinsi"])

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Pemetaan Label Encoding Provinsi:")
for k, v in mapping.items():
    print(f"  {k} -> {v}")

### 4.3 Definisi Fitur (X) dan Target (y) serta Normalisasi

In [ ]:
FITUR = ["Luas Panen", "Curah hujan", "Kelembapan", "Suhu rata-rata", "Provinsi_enc"]
TARGET = "Produksi"

X = df_clean[FITUR].values
y = df_clean[TARGET].values.reshape(-1, 1)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel()

print("Bentuk X:", X_scaled.shape, "| Bentuk y:", y_scaled.shape)

## 5. Pembagian Data Latih & Uji

Data dibagi dengan rasio 80:20.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

print(f"Data latih : {X_train.shape[0]} observasi")
print(f"Data uji   : {X_test.shape[0]} observasi")

## 6. Pelatihan Model Random Forest Regressor

Konfigurasi hiperparameter mengikuti modul `ml_engine.py`.


In [ ]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Model Random Forest berhasil dilatih.")

## 7. Evaluasi Model

Performa diukur menggunakan tiga metrik standar regresi: R², RMSE, dan MAE.
Prediksi dikembalikan ke skala asli (ton) melalui *inverse transform* sebelum perhitungan RMSE dan MAE agar interpretasinya bermakna.


In [ ]:
y_pred_scaled = model.predict(X_test)

# Kembalikan ke skala asli (ton)
y_test_real = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
y_pred_real = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

r2 = r2_score(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae = mean_absolute_error(y_test_real, y_pred_real)

print("=== Performa Model pada Data Uji ===")
print(f"R^2  : {r2:.4f}")
print(f"RMSE : {rmse:,.0f} ton")
print(f"MAE  : {mae:,.0f} ton")

> **Catatan:** Nilai metrik dapat sedikit berbeda tergantung versi pustaka dan komposisi pembagian data. Salin nilai aktual yang diperoleh ke dalam artikel ilmiah agar konsisten.

### 7.1 Visualisasi Prediksi vs Aktual

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_real, y_pred_real, alpha=0.7, edgecolor="k", s=50)
lim_min = min(y_test_real.min(), y_pred_real.min())
lim_max = max(y_test_real.max(), y_pred_real.max())
plt.plot([lim_min, lim_max], [lim_min, lim_max], "r--", label="Prediksi sempurna")
plt.xlabel("Produksi Aktual (ton)")
plt.ylabel("Produksi Prediksi (ton)")
plt.title("Prediksi vs Aktual (Data Uji)", fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Analisis Kepentingan Fitur (Feature Importance)

Random Forest secara *native* menyediakan ukuran kepentingan relatif tiap fitur terhadap prediksi.


In [ ]:
importances = model.feature_importances_
nama_fitur = ["Luas Panen", "Curah Hujan", "Kelembapan", "Suhu Rata-rata", "Provinsi"]

idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.barh([nama_fitur[i] for i in idx][::-1],
         [importances[i] for i in idx][::-1],
         color="#2E8B57")
plt.xlabel("Importance Score")
plt.title("Kepentingan Fitur — Random Forest", fontsize=12)
plt.tight_layout()
plt.show()

print("Peringkat kepentingan fitur:")
for i in idx:
    print(f"  {nama_fitur[i]:<15}: {importances[i]:.3f}")

## 9. Contoh Proyeksi Skenario Iklim (2021–2025)

Mendemonstrasikan mekanisme proyeksi yang digunakan pada **Mode Proyeksi Cerdas** di aplikasi. Baseline diambil dari rata-rata kondisi iklim tiga tahun terakhir (2018–2020) per provinsi, lalu dimodifikasi sesuai skenario delta persentase yang ditentukan pengguna.


In [ ]:
def proyeksi_skenario(provinsi, delta_suhu=0.0, delta_hujan=0.0,
                      delta_lembap=0.0, delta_luas=0.0):
    """Memprediksi produksi untuk satu provinsi berdasarkan skenario delta (%)."""
    sub = df_clean[df_clean["Provinsi"] == provinsi]
    baseline = sub[sub["Tahun"] >= 2018]  # 3 tahun terakhir

    luas   = baseline["Luas Panen"].mean()   * (1 + delta_luas / 100)
    hujan  = baseline["Curah hujan"].mean()  * (1 + delta_hujan / 100)
    lembap = baseline["Kelembapan"].mean()   * (1 + delta_lembap / 100)
    suhu   = baseline["Suhu rata-rata"].mean()* (1 + delta_suhu / 100)
    prov_e = le.transform([provinsi])[0]

    fitur = np.array([[luas, hujan, lembap, suhu, prov_e]])
    fitur_scaled = scaler_X.transform(fitur)
    pred_scaled = model.predict(fitur_scaled)
    pred = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()[0]
    return pred


# Skenario baseline (tanpa perubahan)
prov = "Sumatera Utara"
base = proyeksi_skenario(prov)
print(f"Proyeksi baseline {prov}: {base:,.0f} ton")

# Skenario pemanasan: suhu +2%, curah hujan -15%
skenario = proyeksi_skenario(prov, delta_suhu=2, delta_hujan=-15)
print(f"Skenario (suhu +2%, hujan -15%) {prov}: {skenario:,.0f} ton")
print(f"Selisih: {(skenario - base) / base * 100:+.2f}%")

### 9.1 Perbandingan Skenario untuk Seluruh Provinsi

In [ ]:
hasil = []
for prov in sorted(df_clean["Provinsi"].unique()):
    base = proyeksi_skenario(prov)
    skn = proyeksi_skenario(prov, delta_suhu=2, delta_hujan=-15)
    hasil.append({
        "Provinsi": prov,
        "Baseline (ton)": base,
        "Skenario (ton)": skn,
        "Perubahan (%)": (skn - base) / base * 100,
    })

df_hasil = pd.DataFrame(hasil)
df_hasil

## 10. Kesimpulan

- Model **Random Forest Regressor** berhasil dilatih untuk memprediksi produksi padi delapan provinsi Sumatera berdasarkan faktor iklim dan luas panen.
- Performa model pada data uji menunjukkan nilai **R², RMSE, dan MAE** yang memadai untuk skala produksi jutaan ton (lihat keluaran Bagian 7).
- Analisis kepentingan fitur menempatkan **Luas Panen** dan **Provinsi** sebagai prediktor dominan, dengan **Curah Hujan** sebagai faktor iklim paling berpengaruh.
- Mekanisme proyeksi skenario memungkinkan simulasi dampak perubahan iklim secara interaktif, yang diintegrasikan ke dalam aplikasi Web GIS melalui modul `ml_engine.py` dan `map_visualizer.py`.

Pipeline ini sepenuhnya *reproducible* — menjalankan ulang seluruh sel akan menghasilkan model dan metrik yang konsisten berkat `random_state=42`.
